## Legacy source diagnostics (read-only)
This notebook is in the SAME repository. Never starts the old crawler; it only compares legacy files and displays legacy CSV tables.

In [ ]:
# Connect to existing code and persistent dataset; does not clone or download
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
CODE_DIR = Path("/content/arxiv-code") if Path("/content/arxiv-code/src").is_dir() else Path.cwd()
DATA_DIR = Path(os.environ.get("ARXIV_DATA_DIR",
    "/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data")).resolve()
if not (DATA_DIR / "papers.jsonl").is_file():
    raise FileNotFoundError(f"Missing metadata: {DATA_DIR}")
os.environ["ARXIV_DATA_DIR"] = str(DATA_DIR)
sys.path.insert(0, str(CODE_DIR))
from src import config
assert Path(config.DEFAULT_DATA_DIR).resolve() == DATA_DIR
print("Legacy diagnostics:", DATA_DIR)


### Compare legacy unversioned source to hash-verified vN source

In [ ]:
# No moves, no migration, no deletes
from src.scripts.audit_legacy_matches import audit_legacy_matches
LEGACY_COMPARE_LIMIT = 20
legacy_report = audit_legacy_matches(config.DEFAULT_DATA_DIR,
    max_papers=LEGACY_COMPARE_LIMIT, sample_size=10)
print(legacy_report["summary"])
for row in legacy_report.get("rows", [])[:10]:
    print(row)


### View previous four tables, without rewriting them

In [ ]:
import pandas as pd
from IPython.display import display
for name in ("m1_discovery_stats.csv", "m1_download_stats.csv",
             "m1_reference_stats.csv", "m1_system_benchmark.csv"):
    path = config.DEFAULT_DATA_DIR / "csv" / name
    if path.is_file():
        print(name)
        display(pd.read_csv(path).head(5))
    else:
        print(name, "missing (do not treat as zero)")
